# Week 9 — Monday Lab Notebook  
## Trees

This lab is for **Week 9, Monday** of your course plan.  
Today we will learn **Decision Trees** in simple and practical way.

We will cover:

- what a decision tree is
- root, nodes, branches, and leaves
- how trees split data
- **Gini** and **Entropy** intuition
- depth control
- overfitting and underfitting in trees
- bias and variance idea
- how to read and interpret a tree
- feature importance

The notebook is written in **easy student-friendly language** and is designed for lab teaching in class.

## 3-Hour Structure

### Hour 1
- What is a decision tree?
- Tree structure: root, node, branch, leaf
- Create a dataset
- Features and target
- Train/test split
- Train first decision tree

### Hour 2
- Visualize the tree
- Understand splits
- Gini intuition
- Entropy intuition
- Compare Gini and Entropy

### Hour 3
- Depth control
- Overfitting vs underfitting
- Bias/variance intuition
- Feature importance
- Mini in-class practice
- Recap and tasks

## Part A — What is a Decision Tree?

A **Decision Tree** is a machine learning model that makes decisions by asking a series of questions.

It works like this:

- Is attendance greater than 70?
- Are study hours greater than 4?
- Are previous marks greater than 50?

Based on the answers, the model moves through branches and finally reaches a result.

### Easy idea
A decision tree is like a **flowchart**.

It keeps asking questions until it reaches a final class.

### Example
Suppose we want to predict if a student will **pass** or **fail**.

A tree may ask:

1. Is attendance > 70?
2. If yes, is study hours > 4?
3. If yes, predict **Pass**
4. Otherwise predict **Fail**

## Part B — Main Parts of a Tree

### 1. Root Node
The starting question of the tree.

### 2. Internal Node
A question inside the tree.

### 3. Branch
A path showing the answer to a question.

### 4. Leaf Node
The final output or prediction.

### Memory tip
- **Root** = start
- **Node** = question
- **Branch** = path
- **Leaf** = answer

## Part C — Why Are Decision Trees Popular?

Decision trees are popular because:

- they are easy to understand
- they can work with non-linear patterns
- they do not need heavy mathematical understanding to begin
- they can handle both classification and regression
- they are easy to explain to students and non-technical people

### But they also have weakness
A deep tree can memorize the training data too much.  
This is called **overfitting**.

## Part D — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

## Part E — Create a Simple Student Pass/Fail Dataset

We will make a simple dataset with these features:

- `study_hours`
- `attendance`
- `assignments_submitted`
- `previous_marks`

### Target
- `pass_exam`
    - 1 means pass
    - 0 means fail

This is a teaching dataset.  
It is small, but very useful for learning tree behavior.

In [ ]:
students = pd.DataFrame({
    "study_hours": [1,2,2,3,3,4,4,5,5,6,6,7,7,8,2,1,3,4,6,7,8,5,3,2,9,8,7,6,4,5],
    "attendance": [45,50,55,60,62,68,70,72,75,78,80,82,84,88,52,48,58,65,76,79,90,73,61,54,92,87,83,77,69,74],
    "assignments_submitted": [1,1,2,2,3,3,4,4,5,5,5,6,6,6,2,1,2,3,5,5,6,4,2,2,6,6,5,5,3,4],
    "previous_marks": [30,35,38,42,45,50,55,58,60,64,66,70,72,78,36,32,40,48,65,69,80,57,44,39,84,76,71,67,52,59],
    "pass_exam": [0,0,0,0,0,1,1,1,1,1,1,1,1,1,0,0,0,1,1,1,1,1,0,0,1,1,1,1,1,1]
})

students

## Part F — First Look at the Data

In [ ]:
print(students.head())
print()
print(students.info())

In [ ]:
students.describe()

### What should students notice?

- Every row is one student.
- Input columns are called **features**.
- Output column is called **target**.
- The tree will learn question-like rules from the feature columns.

## Part G — Separate Features and Target

In [ ]:
X = students[["study_hours", "attendance", "assignments_submitted", "previous_marks"]]
y = students["pass_exam"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## Part H — Train/Test Split

We split the data so we can train on one part and test on unseen data.

This helps us answer:

> Is the model learning general rules, or only memorizing training data?

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print()
print("Training class counts:")
print(y_train.value_counts())
print()
print("Testing class counts:")
print(y_test.value_counts())

## Part I — Train the First Decision Tree

We will start with a simple tree using:

- `criterion="gini"`
- `max_depth=3`

### Why limit depth?
If depth becomes too large, the tree may overfit.

In [ ]:
tree_gini = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    random_state=42
)

tree_gini.fit(X_train, y_train)

y_pred_gini = tree_gini.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_gini))

## Part J — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_gini)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fail", "Pass"])
disp.plot()
plt.show()

## Part K — Visualize the Decision Tree

This is one of the best things about decision trees.

We can actually **see** the questions asked by the model.

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    tree_gini,
    feature_names=X.columns,
    class_names=["Fail", "Pass"],
    filled=True,
    rounded=True
)
plt.show()

## Part L — Read the Tree Rules in Text Form

Sometimes a text version is easier to explain in class.

In [ ]:
rules = export_text(tree_gini, feature_names=list(X.columns))
print(rules)

## Part M — How to Interpret a Split

Suppose the tree says:

- `attendance <= 66.5`

This means:

- if attendance is **66.5 or less**, go left
- if attendance is **greater than 66.5**, go right

Each split tries to separate the classes in better way.

### Easy interpretation idea
The tree is trying to ask the question that makes the groups more pure.

## Part N — What Does "Pure" Mean?

A node is **pure** if most values inside it belong to one class.

### Example 1
If a node contains:

- Pass
- Pass
- Pass
- Pass

This node is perfectly pure.

### Example 2
If a node contains:

- Pass
- Fail
- Pass
- Fail

This node is mixed, so purity is lower.

Decision trees try to make child nodes more pure than the parent node.

## Part O — Gini Intuition

**Gini Impurity** tells us how mixed a node is.

- lower Gini = better purity
- higher Gini = more mixed classes

### Easy case
If all values belong to one class:
- Gini = 0

That is perfect purity.

### General formula idea
For two classes:

Gini = 1 - (p1² + p2²)

where:
- `p1` = proportion of class 1
- `p2` = proportion of class 2

You do not need to memorize the formula deeply at first.
Just remember:

> **Lower Gini = better split**

In [ ]:
def gini_impurity(labels):
    values, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()
    return 1 - np.sum(probs ** 2)

node_a = np.array([1,1,1,1,1])          # pure node
node_b = np.array([1,1,1,0,0])          # mixed node
node_c = np.array([1,0,1,0,1,0])        # more mixed

print("Gini of pure node:", round(gini_impurity(node_a), 3))
print("Gini of mixed node:", round(gini_impurity(node_b), 3))
print("Gini of more mixed node:", round(gini_impurity(node_c), 3))

## Part P — Entropy Intuition

**Entropy** is another way to measure impurity.

- low entropy = more pure
- high entropy = more mixed

### Easy memory
Entropy measures **disorder**.

If a node contains only one class, disorder is low.  
If it contains many mixed classes, disorder is high.

For classification trees, sklearn allows:

- `criterion="gini"`
- `criterion="entropy"`

Both try to create better splits.

In [ ]:
def entropy_impurity(labels):
    values, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

print("Entropy of pure node:", round(entropy_impurity(node_a), 3))
print("Entropy of mixed node:", round(entropy_impurity(node_b), 3))
print("Entropy of more mixed node:", round(entropy_impurity(node_c), 3))

### What should students notice?

- Pure node gives **0** impurity.
- Mixed nodes give higher impurity values.
- Gini and Entropy are different formulas.
- But both are used for the same goal:

> make child nodes more pure

## Part Q — Train a Tree with Entropy

Now we will train another tree using `criterion="entropy"` and compare it with Gini.

In [ ]:
tree_entropy = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=3,
    random_state=42
)

tree_entropy.fit(X_train, y_train)

y_pred_entropy = tree_entropy.predict(X_test)

print("Entropy Tree Accuracy:", accuracy_score(y_test, y_pred_entropy))

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Decision Tree (Gini)", "Decision Tree (Entropy)"],
    "Test Accuracy": [
        accuracy_score(y_test, y_pred_gini),
        accuracy_score(y_test, y_pred_entropy)
    ]
})

comparison

## Part R — Gini vs Entropy in Simple Words

### Gini
- usually a little faster
- often used as default
- measures impurity

### Entropy
- based on information disorder
- sometimes gives slightly different splits
- also measures impurity

### Important point
In many practical problems, both work well.  
The difference is often small.

## Part S — Depth Control

`max_depth` controls how deep the tree can grow.

### Small depth
- simple model
- may miss patterns
- may **underfit**

### Large depth
- complex model
- may memorize training data
- may **overfit**

So depth control is very important in decision trees.

In [ ]:
depth_results = []

for depth in [1, 2, 3, 4, None]:
    model = DecisionTreeClassifier(
        criterion="gini",
        max_depth=depth,
        random_state=42
    )
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    depth_results.append({
        "max_depth": str(depth),
        "train_accuracy": accuracy_score(y_train, train_pred),
        "test_accuracy": accuracy_score(y_test, test_pred)
    })

depth_df = pd.DataFrame(depth_results)
depth_df

## Part T — Plot Train Accuracy vs Test Accuracy

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(depth_df["max_depth"], depth_df["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(depth_df["max_depth"], depth_df["test_accuracy"], marker="o", label="Test Accuracy")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Depth Control in Decision Trees")
plt.legend()
plt.grid(True)
plt.show()

## Part U — What Does This Table Mean?

When depth increases:

- training accuracy usually increases
- test accuracy may improve only to a point
- after that, test accuracy may stop improving or may fall

That is a sign of **overfitting**.

### Easy idea
A very deep tree can remember small details of training data.  
But those details may not help on new data.

## Part V — Underfitting vs Overfitting

### Underfitting
The model is too simple.

Example:
- `max_depth = 1`

Possible result:
- low train accuracy
- low test accuracy

### Good fit
The model captures useful patterns.

Example:
- `max_depth = 2` or `3`

Possible result:
- good train accuracy
- good test accuracy

### Overfitting
The model is too complex.

Example:
- `max_depth = None`

Possible result:
- very high train accuracy
- lower test accuracy on new data

## Part W — Bias and Variance Intuition

### High Bias
- model is too simple
- misses important patterns
- underfitting happens

### High Variance
- model is too sensitive to training data
- changes a lot with small data differences
- overfitting happens

### Easy memory
- **High bias = too simple**
- **High variance = too complex**

Decision trees with very large depth often have **high variance**.

## Part X — Feature Importance

Decision trees can tell us which features were most useful for making splits.

This is called **feature importance**.

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": tree_gini.feature_importances_
}).sort_values("Importance", ascending=False)

importance_df

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(importance_df["Feature"], importance_df["Importance"])
plt.title("Feature Importance from Decision Tree")
plt.xlabel("Feature")
plt.ylabel("Importance")
plt.xticks(rotation=45)
plt.show()

### How to explain feature importance in class

If a feature has higher importance, it means the tree used that feature more effectively in important splits.

But remember:

- importance does not always mean cause
- importance values depend on the dataset
- in small datasets, values can change easily

## Part Y — Make Predictions for New Students

In [ ]:
new_students = pd.DataFrame({
    "study_hours": [2, 5, 7],
    "attendance": [55, 72, 86],
    "assignments_submitted": [2, 4, 6],
    "previous_marks": [40, 58, 74]
})

new_predictions = tree_gini.predict(new_students)

result = new_students.copy()
result["predicted_pass_exam"] = new_predictions
result["predicted_label"] = result["predicted_pass_exam"].map({0: "Fail", 1: "Pass"})
result

## Part Z — One More Tree Visualization with More Depth

Let us visualize a deeper tree to see how complexity increases.

In [ ]:
tree_deeper = DecisionTreeClassifier(
    criterion="gini",
    max_depth=4,
    random_state=42
)

tree_deeper.fit(X_train, y_train)

plt.figure(figsize=(18, 9))
plot_tree(
    tree_deeper,
    feature_names=X.columns,
    class_names=["Fail", "Pass"],
    filled=True,
    rounded=True
)
plt.show()

### What should students notice?

- A deeper tree has more questions.
- More questions can mean better fit on training data.
- But too many questions can make the model unstable on new data.

So we do not always want the deepest tree.

## Part AA — Small Real-Life Example 1: Loan Approval

Features can be:

- income
- credit score
- loan amount
- job status

Target:
- approved or rejected

A tree may ask:

- Is credit score > 650?
- Is income > 50000?

## Part AB — Small Real-Life Example 2: Disease Prediction

Features can be:

- age
- sugar level
- blood pressure
- symptoms score

Target:
- disease or no disease

## Part AC — Small Real-Life Example 3: Customer Churn

Features can be:

- monthly bill
- contract type
- support calls
- account age

Target:
- churn or not churn

## Part AD — Important Terms for Students

Students should remember these terms:

- decision tree
- root node
- internal node
- branch
- leaf node
- split
- impurity
- Gini
- Entropy
- max depth
- overfitting
- underfitting
- bias
- variance
- feature importance

## Part AE — Common Beginner Mistakes

### Mistake 1
Thinking deeper tree is always better.

### Mistake 2
Judging model only on training accuracy.

### Mistake 3
Not splitting data into train and test.

### Mistake 4
Using the target column inside features by mistake.

### Mistake 5
Reading the tree figure without understanding the split condition.

### Mistake 6
Thinking feature importance means exact real-world cause.

### Mistake 7
Ignoring `max_depth` and letting the tree grow too much.

## Part AF — Mini In-Class Practice

Ask students to do these in class:

1. Display first 5 rows of the dataset.
2. Separate features and target.
3. Split the data into train and test.
4. Train a decision tree using `criterion="gini"` and `max_depth=2`.
5. Predict on test data.
6. Show test accuracy.
7. Visualize the tree.
8. Print the text rules using `export_text()`.
9. Train another tree with `criterion="entropy"`.
10. Compare train and test accuracy for depths `1`, `2`, and `3`.

## Part AG — Recap

Today we learned:

- what a decision tree is
- root, node, branch, and leaf
- how a tree makes splits
- Gini and Entropy intuition
- how to train and visualize a decision tree
- how to interpret tree rules
- why depth control matters
- underfitting vs overfitting
- bias and variance intuition
- feature importance

### Basic tree workflow
1. prepare data
2. split data
3. train tree
4. predict
5. evaluate
6. control depth
7. interpret results

# After-Lab Tasks (10)

### Task 1
Create a DataFrame of 15 students with:
- study hours
- attendance
- assignments submitted
- previous marks
- pass/fail target

### Task 2
Separate features and target from your dataset.

### Task 3
Split the dataset into training and testing data.

### Task 4
Train a Decision Tree using:
- `criterion="gini"`
- `max_depth=2`

### Task 5
Make predictions on the test data.

### Task 6
Calculate the test accuracy.

### Task 7
Visualize the decision tree using `plot_tree()`.

### Task 8
Print the tree rules using `export_text()` and explain any 2 rules in words.

### Task 9
Train another tree using `criterion="entropy"` and compare it with the Gini tree.

### Task 10
Create a table for depths `1`, `2`, `3`, and `None` showing:
- train accuracy
- test accuracy

Then write 3 lines about underfitting and overfitting.

# Optional Homework Challenge

Create a new decision tree project on one of these topics:

- loan approval
- disease prediction
- employee promotion
- customer churn
- product purchase prediction

### Requirements

1. Make at least 25 rows.
2. Use at least 4 features.
3. Add a binary target column.
4. Split the data into train and test.
5. Train a tree with `criterion="gini"`.
6. Train another tree with `criterion="entropy"`.
7. Try at least 3 different depth values.
8. Compare train and test accuracy.
9. Visualize one tree.
10. Write short observations about:
   - best depth
   - overfitting or underfitting
   - most important feature